In [ ]:
"""
DJI Video SRT Trajectory Mapper - Enhanced with KMZ Output
===========================================================
Reads DJI-style .SRT files (subtitle text files) from drone videos and creates:
1. Interactive HTML web map with Esri Hybrid basemap (Leaflet)
2. KMZ file for Google Earth with trajectories and start/end markers

Features:
- Parses DJI SRT format to extract GPS coordinates and altitude
- Creates one trajectory (polyline) per video
- Adds start and end markers (same color as trajectory)
- Auto-zooms to extent of all trajectories
- Generates KMZ with 3D altitude data

Author: Farid Javadnejad
Date: 2026-04-24
"""

import os
import re
import zipfile
from datetime import datetime
import folium
from folium import plugins
import simplekml

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input folder containing .srt files
INPUT_FOLDER = r"C:\Users\USFJ139860\OneDrive - WSP O365\Projects\260422 - US82 UAS Photoshoot\Videos"

# Output files (saved in same folder as SRT files)
OUTPUT_HTML = os.path.join(INPUT_FOLDER, "drone_video_trajectories.html")
OUTPUT_KMZ = os.path.join(INPUT_FOLDER, "drone_video_trajectories.kmz")

# Trajectory colors (same as image-based web map)
# Colors are reused cyclically if there are more videos than colors
TRAJECTORY_COLORS = [
    '#FF0000',  # Red
    '#00FF00',  # Green
    '#0000FF',  # Blue
    '#FFFF00',  # Yellow
    '#FF00FF',  # Magenta
    '#00FFFF',  # Cyan
    '#FF8800',  # Orange
    '#8800FF',  # Purple
    '#00FF88',  # Spring Green
    '#FF0088'   # Deep Pink
]

# ============================================================================
# SRT PARSING FUNCTIONS
# ============================================================================

def parse_dji_srt_line(line):
    """
    Parse a single line from DJI SRT format to extract GPS coordinates and metadata.
    
    DJI SRT FORMAT EXAMPLE:
    [latitude: 32.965834] [longitude: -103.349189] [rel_alt: 96.500 abs_alt: 1405.827]
    
    Args:
        line (str): Single line of text from SRT file
    
    Returns:
        dict: Dictionary with latitude, longitude, altitude, timestamp or None if parsing fails
    """
    result = {
        'latitude': None,
        'longitude': None,
        'abs_altitude': None,
        'rel_altitude': None,
        'timestamp': None
    }
    
    try:
        # Extract latitude using regex pattern
        lat_match = re.search(r'\[latitude:\s*([-\d.]+)\]', line)
        if lat_match:
            result['latitude'] = float(lat_match.group(1))
        
        # Extract longitude using regex pattern
        lon_match = re.search(r'\[longitude:\s*([-\d.]+)\]', line)
        if lon_match:
            result['longitude'] = float(lon_match.group(1))
        
        # Extract absolute altitude (altitude above sea level)
        abs_alt_match = re.search(r'abs_alt:\s*([-\d.]+)', line)
        if abs_alt_match:
            result['abs_altitude'] = float(abs_alt_match.group(1))
        
        # Extract relative altitude (altitude above takeoff point)
        rel_alt_match = re.search(r'rel_alt:\s*([-\d.]+)', line)
        if rel_alt_match:
            result['rel_altitude'] = float(rel_alt_match.group(1))
        
        # Extract timestamp (format: YYYY-MM-DD HH:MM:SS.mmm)
        timestamp_match = re.search(r'(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d{3})', line)
        if timestamp_match:
            result['timestamp'] = timestamp_match.group(1)
    
    except Exception as e:
        pass  # Return None values if parsing fails
    
    return result


def parse_srt_file(srt_path):
    """
    Parse an entire DJI SRT file and extract all GPS trajectory points.
    
    DJI SRT FILE STRUCTURE:
    Each subtitle entry contains frame metadata including GPS coordinates.
    This function extracts all GPS points in chronological order.
    
    Args:
        srt_path (str): Path to the .srt file
    
    Returns:
        list: List of dictionaries containing GPS points in chronological order
    """
    trajectory_points = []
    
    try:
        with open(srt_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
            # Split into lines and process each line
            lines = content.split('\n')
            
            for line in lines:
                # Parse line for GPS data
                point = parse_dji_srt_line(line)
                
                # Only add points with valid GPS coordinates
                if point['latitude'] is not None and point['longitude'] is not None:
                    trajectory_points.append(point)
    
    except Exception as e:
        print(f"Error parsing {os.path.basename(srt_path)}: {str(e)}")
    
    return trajectory_points


def process_srt_files(folder_path):
    """
    Process all .srt files in a folder and extract trajectories.
    
    Args:
        folder_path (str): Path to folder containing .srt files
    
    Returns:
        list: List of video trajectories, where each trajectory is a dict with:
              - 'filename': name of the SRT file
              - 'points': list of GPS points
              - 'color': assigned trajectory color
    """
    video_trajectories = []
    
    print(f"Scanning folder: {folder_path}")
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return video_trajectories
    
    # Find all .srt files in folder (not in subfolders)
    srt_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.srt')]
    
    if not srt_files:
        print(f"WARNING: No .srt files found in {folder_path}")
        return video_trajectories
    
    print(f"Found {len(srt_files)} .srt file(s)")
    
    # Process each SRT file
    for idx, filename in enumerate(sorted(srt_files)):
        file_path = os.path.join(folder_path, filename)
        print(f"\nProcessing: {filename}")
        
        # Parse SRT file
        points = parse_srt_file(file_path)
        
        if points:
            # Assign color (cyclical reuse if needed)
            color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)]
            
            # Create trajectory object
            trajectory = {
                'filename': filename,
                'points': points,
                'color': color,
                'video_id': idx
            }
            
            video_trajectories.append(trajectory)
            
            print(f"  ✓ Extracted {len(points)} GPS points")
            print(f"  ✓ Start: Lat {points[0]['latitude']:.6f}, Lon {points[0]['longitude']:.6f}")
            print(f"  ✓ End: Lat {points[-1]['latitude']:.6f}, Lon {points[-1]['longitude']:.6f}")
            print(f"  ✓ Assigned color: {color}")
        else:
            print(f"  ✗ No valid GPS points found")
    
    print(f"\nTotal videos with trajectories: {len(video_trajectories)}")
    return video_trajectories


# ============================================================================
# MAP GENERATION
# ============================================================================

def create_trajectory_map(video_trajectories, output_file):
    """
    Create an interactive Leaflet map with video trajectories.
    
    MAP FEATURES:
    - Esri Hybrid basemap (satellite + labels) as DEFAULT
    - One colored polyline per video trajectory
    - Start and end markers using SAME color as trajectory
    - Auto-zoom to extent of all trajectories
    
    Args:
        video_trajectories (list): List of video trajectory objects
        output_file (str): Output HTML file path
    """
    if not video_trajectories:
        print("No trajectories to map!")
        return
    
    # Calculate map center and bounds for all trajectories
    all_lats = []
    all_lons = []
    
    for trajectory in video_trajectories:
        for point in trajectory['points']:
            all_lats.append(point['latitude'])
            all_lons.append(point['longitude'])
    
    # Calculate bounds (southwest and northeast corners)
    min_lat = min(all_lats)
    max_lat = max(all_lats)
    min_lon = min(all_lons)
    max_lon = max(all_lons)
    
    # Calculate center point
    avg_lat = sum(all_lats) / len(all_lats)
    avg_lon = sum(all_lons) / len(all_lons)
    
    # Create base map
    m = folium.Map(
        location=[avg_lat, avg_lon],
        zoom_start=16,
        tiles=None
    )
    
    # ========================================================================
    # BASEMAP CONFIGURATION - ESRI HYBRID (DEFAULT)
    # ========================================================================
    
    # Layer 1: Esri World Imagery (satellite base)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri Satellite',
        overlay=False,
        control=True,
        show=True  # DEFAULT visible basemap
    ).add_to(m)
    
    # Layer 2: Esri Reference Labels - Part of Hybrid
    folium.TileLayer(
        tiles='https://services.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri Labels',
        overlay=True,
        control=False,
        show=True
    ).add_to(m)
        
    # OpenStreetMap (street map)
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='Street Map',
        overlay=False,
        control=True,
        show=False
    ).add_to(m)
    
    # ========================================================================
    # GOOGLE HYBRID - NOT SUPPORTED IN FOLIUM
    # Google Hybrid is NOT included because Folium does not natively support
    # Google Maps JavaScript API with API key integration.
    # RECOMMENDATION: Esri Hybrid provides equivalent functionality.
    # ========================================================================
    
    # ========================================================================
    # ADD TRAJECTORIES WITH SAME-COLOR START/END MARKERS
    # ========================================================================
    
    for trajectory in video_trajectories:
        color = trajectory['color']
        filename = trajectory['filename']
        points = trajectory['points']
        
        # Extract coordinates for polyline
        coords = [[p['latitude'], p['longitude']] for p in points]
        
        # Calculate trajectory statistics
        start_point = points[0]
        end_point = points[-1]
        
        if start_point['abs_altitude'] and end_point['abs_altitude']:
            avg_altitude = sum([p['abs_altitude'] for p in points if p['abs_altitude']]) / len([p for p in points if p['abs_altitude']])
            min_altitude = min([p['abs_altitude'] for p in points if p['abs_altitude']])
            max_altitude = max([p['abs_altitude'] for p in points if p['abs_altitude']])
        else:
            avg_altitude = min_altitude = max_altitude = 0
        
        # Create trajectory polyline
        folium.PolyLine(
            coords,
            color=color,
            weight=3,
            opacity=0.8,
            name=f'{filename} ({len(points)} points)',
            popup=folium.Popup(f"""
                <div style="width: 280px;">
                    <h4 style="margin-bottom: 10px;">{filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Total Points:</b></td><td>{len(points)}</td></tr>
                        <tr><td><b>Avg Altitude:</b></td><td>{avg_altitude:.1f} m</td></tr>
                        <tr><td><b>Altitude Range:</b></td><td>{min_altitude:.1f} - {max_altitude:.1f} m</td></tr>
                        <tr><td><b>Start Time:</b></td><td>{start_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>End Time:</b></td><td>{end_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>Trajectory Color:</b></td><td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
                    </table>
                </div>
            """, max_width=300)
        ).add_to(m)
        
        # ====================================================================
        # START MARKER - Uses SAME color as trajectory
        # Visual distinction: Labeled as "START" in tooltip and popup
        # ====================================================================
        folium.CircleMarker(
            location=[start_point['latitude'], start_point['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: {color};">▶ START: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{start_point['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{start_point['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{start_point['abs_altitude']:.1f} m</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{start_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>Color:</b></td><td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"▶ START: {filename}",
            color='white',
            fillColor=color,  # SAME color as trajectory
            fillOpacity=0.9,
            weight=4  # Thicker outline for visual distinction
        ).add_to(m)
        
        # ====================================================================
        # END MARKER - Uses SAME color as trajectory
        # Visual distinction: Labeled as "END" in tooltip and popup
        # ====================================================================
        folium.CircleMarker(
            location=[end_point['latitude'], end_point['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: {color};">■ END: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{end_point['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{end_point['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{end_point['abs_altitude']:.1f} m</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{end_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>Color:</b></td><td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"■ END: {filename}",
            color='white',
            fillColor=color,  # SAME color as trajectory
            fillOpacity=0.9,
            weight=4  # Thicker outline for visual distinction
        ).add_to(m)
    
    # ========================================================================
    # AUTO-ZOOM TO EXTENT OF ALL TRAJECTORIES
    # ========================================================================
    # Use fit_bounds to automatically zoom and pan the map so that ALL
    # trajectories and markers are visible when the map loads.
    # Bounds are defined as [[southwest_lat, southwest_lon], [northeast_lat, northeast_lon]]
    # ========================================================================
    
    bounds = [[min_lat, min_lon], [max_lat, max_lon]]
    m.fit_bounds(bounds, padding=[30, 30])  # 30px padding for visual buffer
    
    # Add layer control (basemap switcher)
    folium.LayerControl(position='topright').add_to(m)
    
    # Add fullscreen button
    plugins.Fullscreen().add_to(m)
    
    # Add measurement tool
    plugins.MeasureControl(position='topleft').add_to(m)
    
    # Save map
    m.save(output_file)
    print(f"\n✓ Interactive map saved: {output_file}")
    print(f"  - Default basemap: Esri Hybrid (satellite + labels)")
    print(f"  - Auto-zoomed to extent of all trajectories")
    print(f"  - Total trajectories: {len(video_trajectories)}")


# ============================================================================
# KMZ GENERATION
# ============================================================================

def create_kmz_file(video_trajectories, output_file):
    """
    Create a KMZ file with video trajectories and start/end markers.
    
    KMZ FEATURES:
    - One LineString per video trajectory (3D with altitude)
    - START and END placemarks per trajectory
    - Same colors as web map
    - Altitude mode: absolute (relative to sea level)
    
    Args:
        video_trajectories (list): List of video trajectory objects
        output_file (str): Output KMZ file path
    """
    if not video_trajectories:
        print("No trajectories to export to KMZ!")
        return
    
    # Create KML document
    kml = simplekml.Kml()
    
    # Create folders
    trajectories_folder = kml.newfolder(name='Video Trajectories')
    start_markers_folder = kml.newfolder(name='Start Markers')
    end_markers_folder = kml.newfolder(name='End Markers')
    
    for trajectory in video_trajectories:
        color = trajectory['color']
        filename = trajectory['filename']
        points = trajectory['points']
        
        # ====================================================================
        # CREATE TRAJECTORY LINESTRING (3D with altitude)
        # ====================================================================
        # Build coordinates list: (longitude, latitude, altitude)
        # Prefer abs_altitude; fallback to rel_altitude if abs_altitude is missing
        coords = []
        for point in points:
            lon = point['longitude']
            lat = point['latitude']
            # Use absolute altitude if available; otherwise use relative altitude; otherwise 0
            alt = point['abs_altitude'] if point['abs_altitude'] is not None else (point['rel_altitude'] if point['rel_altitude'] is not None else 0)
            coords.append((lon, lat, alt))
        
        # Create LineString
        linestring = trajectories_folder.newlinestring(name=filename)
        linestring.coords = coords
        
        # Convert hex color to KML color format (AABBGGRR)
        hex_color = color.lstrip('#')
        kml_color = f'ff{hex_color[4:6]}{hex_color[2:4]}{hex_color[0:2]}'
        linestring.style.linestyle.color = kml_color
        linestring.style.linestyle.width = 4
        
        # Set altitude mode to absolute (relative to sea level)
        linestring.altitudemode = simplekml.AltitudeMode.absolute
        
        # Add description
        start_point = points[0]
        end_point = points[-1]
        linestring.description = f"""
        <![CDATA[
        <b>Video: {filename}</b><br/>
        Total Points: {len(points)}<br/>
        Color: <span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span><br/>
        Start Time: {start_point['timestamp'] or 'N/A'}<br/>
        End Time: {end_point['timestamp'] or 'N/A'}
        ]]>
        """
        
        # ====================================================================
        # CREATE START MARKER
        # Uses SAME color as trajectory
        # ====================================================================
        start_alt = start_point['abs_altitude'] if start_point['abs_altitude'] is not None else (start_point['rel_altitude'] if start_point['rel_altitude'] is not None else 0)
        
        start_marker = start_markers_folder.newpoint(name=f"START: {filename}")
        start_marker.coords = [(start_point['longitude'], start_point['latitude'], start_alt)]
        start_marker.altitudemode = simplekml.AltitudeMode.absolute
        
        # Use colored icon (Google Earth default pin with color)
        start_marker.style.iconstyle.color = kml_color
        start_marker.style.iconstyle.scale = 1.2
        start_marker.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/paddle/go.png'
        
        start_marker.description = f"""
        <![CDATA[
        <b>START: {filename}</b><br/>
        Latitude: {start_point['latitude']:.6f}°<br/>
        Longitude: {start_point['longitude']:.6f}°<br/>
        Altitude: {start_alt:.1f} m<br/>
        Timestamp: {start_point['timestamp'] or 'N/A'}<br/>
        Color: <span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span>
        ]]>
        """
        
        # ====================================================================
        # CREATE END MARKER
        # Uses SAME color as trajectory
        # ====================================================================
        end_alt = end_point['abs_altitude'] if end_point['abs_altitude'] is not None else (end_point['rel_altitude'] if end_point['rel_altitude'] is not None else 0)
        
        end_marker = end_markers_folder.newpoint(name=f"END: {filename}")
        end_marker.coords = [(end_point['longitude'], end_point['latitude'], end_alt)]
        end_marker.altitudemode = simplekml.AltitudeMode.absolute
        
        # Use colored icon (Google Earth default pin with color)
        end_marker.style.iconstyle.color = kml_color
        end_marker.style.iconstyle.scale = 1.2
        end_marker.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/paddle/stop.png'
        
        end_marker.description = f"""
        <![CDATA[
        <b>END: {filename}</b><br/>
        Latitude: {end_point['latitude']:.6f}°<br/>
        Longitude: {end_point['longitude']:.6f}°<br/>
        Altitude: {end_alt:.1f} m<br/>
        Timestamp: {end_point['timestamp'] or 'N/A'}<br/>
        Color: <span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span>
        ]]>
        """
    
    # Save KML to temporary location
    temp_kml_path = output_file.replace('.kmz', '_temp.kml')
    kml.save(temp_kml_path)
    
    # ========================================================================
    # CREATE KMZ (ZIP ARCHIVE CONTAINING KML)
    # ========================================================================
    # KMZ is simply a ZIP file containing a KML file named "doc.kml"
    # This allows for better compression and potential embedding of resources
    # ========================================================================
    
    with zipfile.ZipFile(output_file, 'w', zipfile.ZIP_DEFLATED) as kmz:
        # Add KML file as doc.kml (required name for KMZ)
        kmz.write(temp_kml_path, 'doc.kml')
    
    # Clean up temporary KML file
    os.remove(temp_kml_path)
    
    print(f"✓ KMZ file saved: {output_file}")
    print(f"  - {len(video_trajectories)} trajectories with 3D altitude data")
    print(f"  - START and END markers for each trajectory")
    print(f"  - Altitude mode: absolute (relative to sea level)")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function.
    """
    print("=" * 70)
    print("DJI VIDEO SRT TRAJECTORY MAPPER")
    print("=" * 70)
    print()
    
    # Process all SRT files in folder
    video_trajectories = process_srt_files(INPUT_FOLDER)
    
    if not video_trajectories:
        print("\n⚠ No video trajectories found. Please check:")
        print("  1. Folder path is correct")
        print("  2. Folder contains .srt files")
        print("  3. SRT files are in DJI format with GPS data")
        return
    
    # Generate outputs
    print("\n" + "=" * 70)
    print("GENERATING OUTPUTS")
    print("=" * 70)
    
    create_trajectory_map(video_trajectories, OUTPUT_HTML)
    create_kmz_file(video_trajectories, OUTPUT_KMZ)
    
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE!")
    print("=" * 70)
    print(f"\nOutputs created:")
    print(f"  • HTML Map: {OUTPUT_HTML}")
    print(f"  • KMZ File: {OUTPUT_KMZ}")
    print(f"\nTotal videos mapped: {len(video_trajectories)}")
    print(f"\nColor assignments:")
    for traj in video_trajectories:
        print(f"  • {traj['filename']}: {traj['color']}")


if __name__ == "__main__":
    main()

DJI VIDEO SRT TRAJECTORY MAPPER

Scanning folder: C:\Users\USFJ139860\OneDrive - WSP O365\Projects\260422 - US82 UAS Photoshoot\Videos
Found 10 .srt file(s)

Processing: DJI_20260422114224_0077_D.SRT
  ✓ Extracted 1766 GPS points
  ✓ Start: Lat 32.965834, Lon -103.349189
  ✓ End: Lat 32.959997, Lon -103.348558
  ✓ Assigned color: #FF0000

Processing: DJI_20260422125704_0008_D.SRT
  ✓ Extracted 614 GPS points
  ✓ Start: Lat 32.856464, Lon -103.763175
  ✓ End: Lat 32.856599, Lon -103.761098
  ✓ Assigned color: #00FF00

Processing: DJI_20260422125734_0009_D.SRT
  ✓ Extracted 68 GPS points
  ✓ Start: Lat 32.856596, Lon -103.761179
  ✓ End: Lat 32.856596, Lon -103.761178
  ✓ Assigned color: #0000FF

Processing: DJI_20260422125901_0023_D.SRT
  ✓ Extracted 1151 GPS points
  ✓ Start: Lat 32.856415, Lon -103.761239
  ✓ End: Lat 32.856551, Lon -103.763855
  ✓ Assigned color: #FFFF00

Processing: DJI_20260422130602_0078_D.SRT
  ✓ Extracted 1503 GPS points
  ✓ Start: Lat 32.856469, Lon -103.763432